# [재학습 · 2026-10-06] 단일 PPO · EIIE · LSRE-CAAN 재학습

회계 수정 후 환경에서 세 원본 노트북의 셀을 **그대로** 다시 실행한다(GitHub 커밋 `0e86a41`의 코드). 크립토 연쇄 재학습 노트북과 **동시에 다른 Colab 세션에서** 돌려도 된다.

- **`런타임 > 모두 실행`만 누르면 된다.** 끊기면 다시 `모두 실행` — 학습된 모델은 캐시에서 불러온다.
- 모델은 `models/rt1006/`, 결과 CSV는 `data/crypto/rt1006/`에 저장된다.
- 단일 PPO는 30만 스텝이라 가장 오래 걸린다(약 45분).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after):
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(cells[i])
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            r = _ip.run_cell(PATCH)
            if not r.success:
                raise RuntimeError('경로 변경 실패')
print('실행기 준비 완료')


In [ ]:
# 단일 정책 PPO (표 1의 행 1)
run_cells('crypto/notebooks/crypto_rl_trading_colab.ipynb', [2, 4, 6, 8, 9, 11, 13, 15, 16], patch_after=2)


In [ ]:
# EIIE (6.6절)
run_cells('crypto/notebooks/crypto_eiie_baseline_colab.ipynb', [3, 5, 7, 9, 11, 13, 15, 17, 18, 20], patch_after=3)


In [ ]:
# LSRE-CAAN (6.6절)
run_cells('crypto/notebooks/crypto_lsre_caan_baseline_colab.ipynb', [3, 5, 7, 9, 11, 13, 15, 17, 18, 20], patch_after=3)
print('\n전부 완료')
